In [19]:
import pandas as pd
from pathlib import Path
import os
from datetime import datetime,timedelta
today = datetime.today().date()
yesterday = datetime.now() - timedelta(days=2)


In [20]:
def create_error_file(dataframe: pd.DataFrame, filename: str):
    if dataframe.shape[0]>0:
        output_dir = Path("output")
        output_dir.mkdir(exist_ok=True)
        file_path = output_dir / f"[{today} - Anomaly] {filename}.csv"

        dataframe.to_csv(file_path, index=False)
    
    
    

In [21]:
def extract_compoundname(package):
    print(package)
    
    if "-" in package:
        return package.split("-",1)[1]
    elif "El Gouna" in package:
        return "El Gouna"
    else:
        return None


In [22]:
def extract_provider(package):
    package = str(package).lower()

    bein_keywords = [
        "bein",
        "afcon",
        "euro",
        
    ]

    if any(keyword in package for keyword in bein_keywords):
        return "beIN"

    if "osn" in package:
        return "OSN"
    
    if "fta" in package:
        return "FTA"

    return "N/A"

In [23]:
def load_files(path,date):

    folder = Path(path)
    files = [f for f in folder.iterdir() if f.is_file()]
    columns = ["SUBSCRIBE_SERVICE_ID",	"SERVICE_MENU_ID",	"SUBSCRIPTION_DATE",	"EXPIRE_DATE",	"WORK_PHONE",	"CUSTOMER_ID",	"NAME",	"DEVICE_ID",	"MAC_ADDRESS",	"SOURCEFILE"]
    all = pd.DataFrame(columns=columns)
    for index, file in enumerate(files, start=1):
        print(file)
        
        data = pd.read_csv(os.path.join(folder,file.name),dtype='str')
        
        data['SOURCEFILE'] = file.name
        data['COMPOUND'] = data['NAME'].apply(extract_compoundname)
        compound_from_filename  = (
            file.name
            .replace("CNE_", "")
            .replace("Minerva.csv", "")
            .replace("_", "")
        )

        all = pd.concat([all,data],ignore_index=False)
        all['PROVIDER'] = all['NAME'].apply(extract_provider)
        all['REPORT DATE'] = date
        all = all.loc[all['PROVIDER']=='beIN']
        all.to_csv(f'Orange All {date}.csv')
    return all






Create One File for all days

In [24]:

folder_path = Path(r"C:\Users\mturky\Documents\orange data\9-7-2026")

subfolders = [item for item in folder_path.iterdir() if item.is_dir()]

all_results = []

if subfolders:
    # Process each subfolder
    for folder in subfolders:
        all_results.append(
            load_files(folder, folder.name)
        )
else:
    # Process files directly in the root folder
    all_results.append(
        load_files(folder_path, folder_path.name)
    )

final_df = pd.concat(all_results, ignore_index=True)
display(final_df)

final_df['EXPIRE_DATE'] = pd.to_datetime(final_df['EXPIRE_DATE'],dayfirst=True,errors='coerce')
final_df['SUBSCRIPTION_DATE'] = pd.to_datetime(final_df['SUBSCRIPTION_DATE'],dayfirst=True,errors='coerce')


final_df.to_csv(
    "final df.csv",
    index=False
)

C:\Users\mturky\Documents\orange data\9-7-2026\CNE Sodic west Minerva july.csv
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWest
beIN Sports-SodicWe

TypeError: argument of type 'float' is not iterable

In [ ]:

final_df = final_df.loc[final_df['SUBSCRIPTION_DATE'].dt.year==2026]

final_df['SUB_MONTH'] =final_df['SUBSCRIPTION_DATE'].dt.to_period('M')

months = sorted(final_df['SUB_MONTH'].dropna().unique())


new_macs_by_month = []

for month in months:
    current_macs = final_df.loc[
            final_df['SUB_MONTH'] == month,
            'MAC_ADDRESS'
        ]
    

    # new_macs = current_macs - seen

    for mac in current_macs:
        new_macs_by_month.append({
            'SUB_MONTH': str(month),
            'MAC_ADDRESS': mac
        })

    

new_macs_df = pd.DataFrame(new_macs_by_month)




In [ ]:
new_macs_df.to_csv('new_macs.csv',index=False)


Check for missing MAC ADDRESS

In [ ]:
all_old = load_files(r"C:\Users\mturky\Documents\orange data\1-6-2026", '1-6-2026')
all_new = load_files(r"C:\Users\mturky\Documents\orange data\8-6-2026",'8-6-2026')

bein_old = all_old.loc[all_old['NAME'].str.lower().str.contains('bein')]
bein_new = all_new.loc[all_new['NAME'].str.lower().str.contains('bein')]

missing = bein_old.loc[~bein_old['MAC_ADDRESS'].isin(bein_new['MAC_ADDRESS'])]
missing_future_date = missing.loc[missing['EXPIRE_DATE']>'2026-06-01']
missing_empty_date = missing.loc[missing['EXPIRE_DATE'].isna()]

create_error_file (missing_future_date,"Missing with future expire date")
create_error_file(missing_empty_date,"Missing with empty expire date")

In [ ]:
bein_new.columns

Check for missing DUPLICATES

In [ ]:
agg = bein_new.groupby(['WORK_PHONE','MAC_ADDRESS','SOURCEFILE']).agg(count = ('MAC_ADDRESS','count')).reset_index()
agg = agg.loc[agg['count']>1]
agg = agg.astype(str)

create_error_file(agg[['WORK_PHONE','MAC_ADDRESS','SOURCEFILE']], 'Audit sample')

Empty MAC_ADDRESS

In [ ]:
empty_mac = bein_new.loc[bein_new['MAC_ADDRESS'].isna()]
create_error_file(empty_mac,"Empty MAC ADDRESS")

Empty EXPIRE DATE

In [ ]:
empty_expire = bein_new.loc[bein_new['EXPIRE_DATE'].isna()]
create_error_file(empty_expire,"Empty EXPIRE DATE")

Expired Files

In [ ]:
empty_expire = bein_new.loc[bein_new['EXPIRE_DATE']< yesterday ]
create_error_file(empty_expire,"Expired contracts")

In [ ]:
all_new.columns